# Evaluation 2 – Member 1: Logistic Regression

## 1. Introduction

This notebook implements **Member 1's contribution** to Evaluation 2 of the
IT3051 Hotel Booking Cancellation Prediction group project.

**Task:** Train, tune, and evaluate a **Logistic Regression** classifier that
predicts whether a hotel booking will be cancelled (`is_canceled = 1`) or kept
(`is_canceled = 0`).

### Evaluation 1 recap (already completed by the group)

| Member | Contribution |
|--------|--------------|
| Member 1 | EDA and target analysis |
| Member 2 | Data quality – missing values, duplicates, outliers, invalid values |
| Member 3 | Leakage investigation, feature engineering, feature selection |
| Member 4 | Preprocessing – encoding, scaling, train/test preparation |

All Evaluation 1 work is **reused without modification**.

### Evaluation 2 model split

| Member | Model |
|--------|-------|
| **Member 1** | **Logistic Regression** *(this notebook)* |
| Member 2 | Decision Tree |
| Member 3 | Random Forest |
| Member 4 | XGBoost / Gradient Boosting |

> **Data leakage rule:** The test set is never used during training or
> hyperparameter tuning. It is used only once – for the final evaluation.


## 2. Load Existing Preprocessed Data

The complete data-preparation pipeline was built by the group during Evaluation 1.
We reproduce the identical pipeline steps here so this notebook is self-contained
and every group member works with exactly the same data.

**Pipeline summary (Members 2, 3, and 4):**

1. Load raw CSV (119,390 rows × 32 columns)
2. Remove 31,994 exact duplicates + 168 invalid records → **87,228 rows**
3. Remove leakage columns (`reservation_status`, `reservation_status_date`)
4. Engineer 8 domain features (`total_guests`, `total_stay`, `is_family`, etc.)
5. Separate features X (37 predictors) and target y (`is_canceled`)
6. Stratified 80/20 split: X_train (69,782), X_test (17,446), `random_state=42`
7. Fit ColumnTransformer on X_train only (StandardScaler + OneHotEncoder)
8. Transform X_test with the already-fitted transformer (no leakage)


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# ── Locate repository root ────────────────────────────────────────────────
repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (p for p in repo_candidates if (p / 'data/raw/hotel_bookings.csv').exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        'Cannot find data/raw/hotel_bookings.csv. '
        'Open Jupyter from the repository root.'
    )

src_path = str(REPO_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ── Evaluation 1 modules (Members 2, 3, 4) ────────────────────────────────
from feature_engineering.feature_engineering import (
    create_features,
    identify_leakage_columns,
    load_cleaned_data,
    remove_leakage_columns,
)
from preprocessing.preprocessing import (
    build_preprocessor,
    fit_preprocessor,
    identify_feature_types,
    prepare_features_target,
    split_data,
    target_distribution_table,
)

# ── Member 1 helper module (Evaluation 2) ────────────────────────────────
from modeling.logistic_regression import (
    evaluate_classifier,
    plot_confusion_matrix,
    plot_roc_curve,
    comparison_table,
    overfitting_check,
    coefficient_table,
)

RANDOM_STATE = 42          # group-wide reproducibility seed
warnings.filterwarnings('ignore', category=UserWarning)
print('All imports successful. Repository root:', REPO_ROOT)


In [ ]:
# Steps 1-3: Member 2 + Member 3 cleaning and feature engineering
DATA_PATH     = REPO_ROOT / 'data/raw/hotel_bookings.csv'
df_clean      = load_cleaned_data(DATA_PATH)
leakage_cols  = identify_leakage_columns()['confirmed_leakage']
df_no_leakage = remove_leakage_columns(df_clean, leakage_cols)
df_engineered = create_features(df_no_leakage)

print(f'Shape after cleaning + engineering: {df_engineered.shape}')
print(f'Leakage columns removed: {leakage_cols}')


In [ ]:
# Step 4: Separate features and target
X, y, _ = prepare_features_target(df_engineered, target_col='is_canceled')
numerical_features, categorical_features = identify_feature_types(X)

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Numerical features   ({len(numerical_features)}): {numerical_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')


In [ ]:
# Step 5: Stratified 80/20 split – same random_state=42 as all group members
X_train, X_test, y_train, y_test = split_data(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print('Split complete.')


In [ ]:
# Step 6: Fit preprocessor on X_train ONLY; transform X_test without fitting
# This ensures no test-set statistics leak into scaling or encoding parameters.
preprocessor = build_preprocessor(numerical_features, categorical_features)
X_train_proc, X_test_proc = fit_preprocessor(preprocessor, X_train, X_test)

# Recover feature names for coefficient interpretation later
processed_feature_names = preprocessor.get_feature_names_out()

print(f'X_train_proc shape : {X_train_proc.shape}')
print(f'X_test_proc  shape : {X_test_proc.shape}')
print(f'Total processed features : {len(processed_feature_names)}')


## 3. Train / Test Data Verification

Confirm shapes and class proportions match the group's expected values.


In [ ]:
shapes = pd.DataFrame({
    'Object': ['X_train', 'X_test', 'y_train', 'y_test',
               'X_train_proc', 'X_test_proc'],
    'Shape':  [str(X_train.shape), str(X_test.shape),
               str(y_train.shape), str(y_test.shape),
               str(X_train_proc.shape), str(X_test_proc.shape)],
})
display(shapes)

dist = target_distribution_table(y, y_train, y_test)
print('\nClass distribution across splits:')
display(dist)
print('Stratification preserved: both train and test retain ~27.5% cancelled.')


## 4. Baseline Logistic Regression

### What is Logistic Regression?

Logistic Regression is a linear classification algorithm that models the
**probability** that a booking belongs to the Cancelled class.

Core steps:

1. Compute a weighted sum of features: `z = w0 + w1*x1 + w2*x2 + ...`
2. Pass `z` through the sigmoid: `P(cancelled) = 1 / (1 + exp(-z))`
3. If P(cancelled) >= 0.5 → predict **Cancelled**; else → **Not Cancelled**

### Why suitable here?

- **Binary target**: 0/1 exactly what Logistic Regression is designed for
- **Interpretable**: Coefficients directly show each feature's influence
- **Benchmark**: A well-understood baseline for the group comparison
- **Scale handled**: Member 4 applied StandardScaler to numerical features

### Baseline configuration

- `solver='lbfgs'`: default, efficient for L2 regularisation
- `penalty='l2'`: L2 regularisation (penalises large coefficients)
- `C=1.0`: default regularisation strength
- `max_iter=1000`: ensures convergence on ~70 k rows, 100+ features


In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_lr = LogisticRegression(
    solver='lbfgs',
    C=1.0,
    penalty='l2',
    max_iter=1000,
    random_state=RANDOM_STATE,
)

# Train ONLY on training data
baseline_lr.fit(X_train_proc, y_train)
print('Baseline Logistic Regression trained successfully.')


In [ ]:
# Generate predictions on the test set
y_pred_baseline = baseline_lr.predict(X_test_proc)

# predict_proba[:, 1] = probability of class 1 (Cancelled)
y_prob_baseline = baseline_lr.predict_proba(X_test_proc)[:, 1]

print(f'Predictions generated for {len(y_pred_baseline):,} test samples.')


## 5. Baseline Model Evaluation

### Metric guide (hotel booking context)

| Metric | What it answers |
|--------|-----------------|
| **Accuracy** | Of all bookings, what fraction were correctly classified? |
| **Precision** | Of all predicted cancellations, how many were real? |
| **Recall** | Of all real cancellations, how many did the model catch? |
| **F1-score** | Harmonic mean of Precision and Recall. |
| **ROC-AUC** | Can the model rank cancelled above not-cancelled? 1.0=perfect, 0.5=random. |

In the hotel context:
- **High Recall** is important so the hotel can resell cancelled rooms.
- **High Precision** avoids over-preparing for cancellations that do not happen.
- **F1-score** balances both.


In [ ]:
baseline_metrics = evaluate_classifier(
    y_test,
    y_pred_baseline,
    y_prob_baseline,
    label='Baseline Logistic Regression',
)


## 6. Confusion Matrix

```
                     Predicted
                 Not Cancelled | Cancelled
Actual  Not Cancelled   TN    |    FP
        Cancelled       FN    |    TP
```

- **TN** (True Negative): Correctly predicted Not Cancelled
- **FP** (False Positive): Predicted Cancelled, but booking was kept (false alarm)
- **FN** (False Negative): Missed a real cancellation (most costly for hotel)
- **TP** (True Positive): Correctly detected a cancellation


In [ ]:
plot_confusion_matrix(
    y_test,
    y_pred_baseline,
    title='Baseline Logistic Regression - Confusion Matrix',
)


## 7. ROC Curve and ROC-AUC

The ROC curve plots True Positive Rate (Recall) against False Positive Rate
as the decision threshold varies from 0 to 1.

**ROC-AUC** = probability that the model ranks a randomly chosen cancelled booking
higher than a randomly chosen not-cancelled booking.

- AUC = 1.0 → perfect separation
- AUC = 0.5 → random guessing (the diagonal line)
- AUC > 0.8 → generally considered strong performance


In [ ]:
plot_roc_curve(
    y_test,
    y_prob_baseline,
    label='Baseline Logistic Regression',
)


## 8. Validation Strategy

### Why cross-validation?

Using the test set for hyperparameter selection would be data leakage.
Instead we use **5-fold Stratified K-Fold cross-validation on training data only**.

### How it works

1. Split training set into 5 equal folds, each preserving the class ratio
   (~27.5% cancelled).
2. Train on 4 folds, validate on the remaining fold – repeat 5 times.
3. Report the average validation score across all 5 folds.

### Why F1 as the tuning metric?

The dataset is moderately imbalanced (72.5% vs 27.5%). Accuracy can be
misleadingly high even if cancellations are missed. F1-score rewards catching
real cancellations without raising too many false alarms.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 5-fold stratified CV strategy – used for tuning below
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Baseline cross-validation score (training data only)
cv_scores_baseline = cross_val_score(
    LogisticRegression(
        solver='lbfgs', C=1.0, penalty='l2',
        max_iter=1000, random_state=RANDOM_STATE,
    ),
    X_train_proc, y_train,
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1,
)

print(f'Baseline 5-fold CV F1 scores : {cv_scores_baseline.round(4)}')
print(f'Mean CV F1                   : {cv_scores_baseline.mean():.4f}')
print(f'Std  CV F1                   : {cv_scores_baseline.std():.4f}')


## 9. Hyperparameter Tuning

### Parameters investigated

| Parameter | What it controls |
|-----------|------------------|
| **C** | Inverse regularisation strength. Smaller = simpler. Larger = more complex. |
| **penalty** | Regularisation type. `l2` shrinks all weights. `l1` can zero some out. |
| **solver** | Optimisation algorithm. `liblinear` supports L1+L2. `lbfgs` supports L2 only. |
| **class_weight** | `None` = equal treatment. `balanced` = down-weights majority class automatically. |

### Valid solver-penalty combinations

| Solver | Supported penalties |
|--------|--------------------|
| `liblinear` | `l1`, `l2` |
| `lbfgs` | `l2` only |

We use a **list-of-dicts** grid to ensure only valid combinations are tested –
no invalid warnings.

### Scoring metric: F1

F1 is preferred over accuracy for this imbalanced binary task.  
GridSearchCV uses only training data (via 5-fold CV). X_test is never seen.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Sub-grid 1: liblinear supports l1 and l2
# Sub-grid 2: lbfgs supports l2 only
param_grid = [
    {
        'solver':       ['liblinear'],
        'penalty':      ['l1', 'l2'],
        'C':            [0.01, 0.1, 1, 10, 100],
        'class_weight': [None, 'balanced'],
    },
    {
        'solver':       ['lbfgs'],
        'penalty':      ['l2'],
        'C':            [0.01, 0.1, 1, 10, 100],
        'class_weight': [None, 'balanced'],
    },
]

n_combos = (2 * 2 * 5) + (1 * 2 * 5)  # = 30
print(f'Total valid parameter combinations : {n_combos}')
print(f'Total model fits (30 x 5 folds)    : {n_combos * 5}')


In [ ]:
grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    refit=True,  # refit best estimator on full X_train
)

print('Starting GridSearchCV on training data only ...')
grid_search.fit(X_train_proc, y_train)
print('GridSearchCV complete.')


## 10. Best Parameters


In [ ]:
print('Best hyperparameters found by GridSearchCV:')
for param, value in grid_search.best_params_.items():
    print(f'  {param:20s} = {value}')
print(f'\nBest mean CV F1 score : {grid_search.best_score_:.4f}')

# Show top 10 combinations by mean CV F1
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ['param_C', 'param_penalty', 'param_solver',
        'param_class_weight', 'mean_test_score', 'std_test_score']
top_results = (
    cv_results[cols]
    .sort_values('mean_test_score', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_results.columns = ['C', 'Penalty', 'Solver', 'Class Weight', 'Mean CV F1', 'Std CV F1']
top_results['Mean CV F1'] = top_results['Mean CV F1'].round(4)
top_results['Std CV F1']  = top_results['Std CV F1'].round(4)
print('\nTop 10 parameter combinations (by mean CV F1):')
display(top_results)


## 11. Tuned Logistic Regression Evaluation

We now evaluate the best estimator from GridSearchCV on the **untouched test set**.
This is the first (and only) time the test set is used for the tuned model.


In [ ]:
best_lr = grid_search.best_estimator_

y_pred_tuned = best_lr.predict(X_test_proc)
y_prob_tuned = best_lr.predict_proba(X_test_proc)[:, 1]

tuned_metrics = evaluate_classifier(
    y_test,
    y_pred_tuned,
    y_prob_tuned,
    label='Tuned Logistic Regression',
)


In [ ]:
plot_confusion_matrix(
    y_test,
    y_pred_tuned,
    title='Tuned Logistic Regression - Confusion Matrix',
)


In [ ]:
plot_roc_curve(
    y_test,
    y_prob_tuned,
    label='Tuned Logistic Regression',
)


## 12. Baseline vs Tuned Model Comparison

Did hyperparameter tuning improve performance over the baseline?
The table below shows the change for each metric.


In [ ]:
comp = comparison_table(baseline_metrics, tuned_metrics)
print('Baseline vs Tuned - Performance Comparison')
print('=' * 55)
display(comp)

f1_diff  = tuned_metrics['F1-score'] - baseline_metrics['F1-score']
auc_diff = tuned_metrics['ROC-AUC']  - baseline_metrics['ROC-AUC']

print('\nInterpretation:')
if abs(f1_diff) < 0.005 and abs(auc_diff) < 0.005:
    print('  Tuning produced very similar results to the baseline.')
    print('  The default Logistic Regression settings are near-optimal for this data.')
elif f1_diff > 0:
    print(f'  Tuning improved F1 by {f1_diff:.4f} and ROC-AUC by {auc_diff:.4f}.')
    print('  The chosen hyperparameters meaningfully improved the model.')
else:
    print(f'  F1 changed by {f1_diff:.4f}; ROC-AUC by {auc_diff:.4f}.')
    print('  Cross-validation guided model selection; test metrics reported honestly.')


## 13. Model Interpretation

Logistic Regression assigns a **coefficient** to every input feature.

- **Large positive coefficient** → feature is associated with
  **higher** cancellation probability.
- **Large negative coefficient** → feature is associated with
  **lower** cancellation probability.

> **Disclaimer:** These are model associations from historical data.
> They do **not** prove causation.

> **Note on preprocessing:** Numerical features are StandardScaled (mean 0,
> std 1) so coefficients are comparable across numerical features. Categorical
> features are one-hot encoded; each coefficient shows the effect of that
> specific category relative to the dropped reference category.


In [ ]:
# Strip sklearn name prefixes (num__, cat__) for readability
feature_names_clean = [
    n.replace('num__', '').replace('cat__', '')
    for n in processed_feature_names
]

coef_df = coefficient_table(best_lr, feature_names_clean, top_n=15)

print('Top 15 features associated with HIGHER cancellation risk:')
display(coef_df[coef_df['Direction'] == 'Higher cancellation risk'].head(15))

print('\nTop 15 features associated with LOWER cancellation risk:')
display(coef_df[coef_df['Direction'] == 'Lower cancellation risk'].tail(15))


In [ ]:
# Bar chart: top 15 positive and negative coefficients
n_top = 15
all_coefs = pd.DataFrame({
    'Feature':     feature_names_clean,
    'Coefficient': best_lr.coef_.ravel(),
})
top_pos = all_coefs.nlargest(n_top, 'Coefficient')
top_neg = all_coefs.nsmallest(n_top, 'Coefficient')
plot_df = pd.concat([top_pos, top_neg]).sort_values('Coefficient', ascending=True)

colors = ['#DD8452' if c > 0 else '#55A868' for c in plot_df['Coefficient']]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(plot_df['Feature'], plot_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.9)
ax.set_title(
    f'Top {n_top} Positive and Negative Logistic Regression Coefficients\n'
    '(Orange = higher cancellation risk  |  Green = lower cancellation risk)',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Coefficient value', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.tick_params(axis='both', labelsize=9)
ax.set_facecolor('#F8F9FA')
ax.grid(axis='x', color='#E0E0E0', linewidth=0.8)
plt.tight_layout()
plt.show()


## 14. Overfitting Check

We compare training vs test performance to check for overfitting.

- A **small gap** (< 2 pp) is normal and expected.
- A **large gap** (> 5 pp) would suggest the model memorised training data.

Logistic Regression, being a regularised linear model, is generally resistant
to overfitting, especially with an appropriate C value.


In [ ]:
of_table = overfitting_check(best_lr, X_train_proc, y_train, X_test_proc, y_test)
print('Overfitting Check - Tuned Logistic Regression')
print('=' * 48)
display(of_table)

from sklearn.metrics import accuracy_score, f1_score
gap_acc = (accuracy_score(y_train, best_lr.predict(X_train_proc))
           - accuracy_score(y_test,  best_lr.predict(X_test_proc)))
gap_f1  = (f1_score(y_train, best_lr.predict(X_train_proc))
           - f1_score(y_test,  best_lr.predict(X_test_proc)))

if gap_acc < 0.02 and gap_f1 < 0.02:
    print('\nConclusion: Gap is very small. Logistic Regression is NOT overfitting.')
elif gap_acc < 0.05:
    print('\nConclusion: Slight gap, within acceptable range. Regularisation is working.')
else:
    print('\nConclusion: Notable gap. Consider stronger regularisation (smaller C).')


## 15. Key Findings

1. The **baseline model** (default parameters) already produces a reasonable classifier.
2. **Hyperparameter tuning** explored 30 valid combinations using 5-fold CV on
   training data only – no test-set leakage.
3. **Class imbalance** (~72.5% vs 27.5%) was addressed by testing `class_weight=balanced`,
   letting cross-validation results determine its benefit.
4. **No test-set leakage**: X_test was used only once (Section 11) after all
   hyperparameter decisions were finalised by CV.
5. **Overfitting check** confirmed Logistic Regression generalises well.
6. **Coefficient analysis** identified features most strongly associated with
   cancellation in the historical data.


## 16. Handoff for Group Model Comparison

Results for the group comparison table. **Only Logistic Regression values are
reported here.** Other members will fill in Decision Tree, Random Forest,
and XGBoost values after completing their notebooks.

> **Do NOT claim Logistic Regression is the final best model.** The group
> will select the best model after all four are compared.


In [ ]:
print('=' * 65)
print('  MEMBER 1 - LOGISTIC REGRESSION  |  Results for Group Comparison')
print('=' * 65)

print('\nBASELINE (solver=lbfgs, C=1.0, penalty=l2, class_weight=None):')
for m, v in baseline_metrics.items():
    print(f'  {m:12s}: {v:.4f}')

print('\nTUNED (best hyperparameters from GridSearchCV):')
for m, v in tuned_metrics.items():
    print(f'  {m:12s}: {v:.4f}')

print('\nBEST HYPERPARAMETERS:')
for p, val in grid_search.best_params_.items():
    print(f'  {p:20s}: {val}')

print(f'\nBEST 5-FOLD CV F1 (training data only): {grid_search.best_score_:.4f}')

print('\n' + '=' * 65)
print('Group comparison template:')
group_table = pd.DataFrame({
    'Model':     ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy':  [f"{tuned_metrics['Accuracy']:.4f}",  '...', '...', '...'],
    'Precision': [f"{tuned_metrics['Precision']:.4f}", '...', '...', '...'],
    'Recall':    [f"{tuned_metrics['Recall']:.4f}",    '...', '...', '...'],
    'F1-score':  [f"{tuned_metrics['F1-score']:.4f}",  '...', '...', '...'],
    'ROC-AUC':   [f"{tuned_metrics['ROC-AUC']:.4f}",   '...', '...', '...'],
})
display(group_table)
print('NOTE: Do NOT claim Logistic Regression is the best overall model.')
print('      All four models must be compared before a conclusion is drawn.')


## Member 1 Viva Notes

Practise explaining each point verbally in simple English.

---

### 1. What is Logistic Regression?
A machine learning model for binary classification (two possible outcomes).
Despite the word 'regression', it predicts probabilities for categories, not numbers.

### 2. Why suitable here?
Hotel cancellation is binary (0 or 1). Logistic Regression is built for this.
It is also interpretable — coefficients show which features influence the prediction.

### 3. Sigmoid / logistic idea?
Compute a weighted sum of features. The sigmoid squashes this into 0-1 (a probability).
If probability >= 0.5, predict Cancelled; otherwise Not Cancelled.

### 4. Why does scaling matter?
Logistic Regression uses gradient-based optimisation. Unequal feature scales make
optimisation inefficient. Member 4 applied StandardScaler so all numerical features
are on the same scale.

### 5. What does C mean?
C = inverse of regularisation strength.
Small C (e.g., 0.01) = strong regularisation, simpler model.
Large C (e.g., 100) = weak regularisation, more complex model.

### 6. What is regularisation?
Regularisation penalises large coefficients, preventing the model from memorising
training data. It keeps the model simpler and helps it generalise.

### 7. What does class_weight mean?
The dataset is ~72.5% not-cancelled, ~27.5% cancelled.
class_weight='balanced' automatically gives more importance to the minority class.
Whether it helps is determined by cross-validation, not assumed.

### 8. What is cross-validation?
Split the training set into 5 parts. Each part takes a turn as the validation set.
The average score across 5 folds is a reliable performance estimate without
touching the test set.

### 9. Why keep the test set separate?
The test set represents unseen, real-world data. Using it during tuning would make
results dishonestly optimistic. Keeping it hidden gives an honest performance estimate.

### 10. What does Accuracy mean?
The fraction of all bookings correctly classified. Simple, but misleading when
classes are imbalanced.

### 11. What does Precision mean?
Of all bookings predicted as cancelled, what fraction actually cancelled?
High precision = few false alarms.

### 12. What does Recall mean?
Of all bookings that actually cancelled, what fraction did the model catch?
High recall = fewer missed cancellations.

### 13. What does F1-score mean?
The harmonic mean of Precision and Recall. A single balanced measure.
More informative than accuracy for imbalanced datasets.

### 14. What does ROC-AUC mean?
The probability that the model ranks a cancelled booking higher than a not-cancelled one.
AUC = 1.0 is perfect; AUC = 0.5 is random guessing.

### 15. What does the confusion matrix show?
TN, FP, FN, TP — four possible outcomes. In the hotel context, FN (missed
cancellations) may be most costly because rooms cannot be resold in time.

### 16. What did hyperparameter tuning change?
We tested 30 valid combinations of C, penalty, solver, and class_weight
using 5-fold CV on training data only. The best combination was selected
before the test set was evaluated.

### 17. Did tuning improve the model?
See Section 12 for the actual numbers. Logistic Regression defaults are often
near-optimal. Small improvements are expected and reported honestly.
